[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/44.%20The%20Carbon%20Footprint%20Modeling%20Problem/P44-Tier-8_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 44. The Carbon Footprint Modeling Problem

## Tier 8: The Value-Aligned and Ethical Framework

### Goal

Evaluate a fixed set of carbon-reduction scenarios using stakeholder-weighted values and ethical constraints. The framework first checks each scenario against hard and soft requirements, then computes a value-aligned score, ranks the scenarios, and reports the highest-scoring option. A hard-constraint violation makes a scenario ineligible by assigning it a score of zero; soft-constraint violations reduce its score through a penalty.

This is a transparent, rule-based decision framework, not a claim that ethical values can be measured without judgment. Its weights, impact scales, thresholds, and scoring rules are explicit modeling choices that decision-makers would need to justify with stakeholders.

### Inputs, outputs, and scope

**Inputs:** Four stakeholder groups and their weights, four ethical constraints, and five scenarios with carbon reductions, cost changes, and stakeholder-impact scores.

**Outputs:** Constraint-compliance assessments, weighted value-aligned scores, a scenario ranking, plots, and a recommendation based on the highest score.

**Scope and assumptions:** Impact scores are supplied scenario data rather than measurements calculated from operational records. The values are treated as comparable scores on a scale that is generally between $-1$ and $1$. Carbon reduction is measured in kg CO$_2$e/day. Cost change is measured as a percentage, where a negative value represents a cost decrease. The scoring rule normalizes carbon reduction by 450 kg CO$_2$e/day, the largest reduction among the scenarios in this example. The model does not estimate emissions, jobs, health outcomes, uncertainty, or causal effects; it also does not model stakeholder deliberation or changes over time. A score is a decision aid, not a probability or an objective measure of ethical worth.

### Modeling approach

A **stakeholder weight** represents the relative importance assigned to a stakeholder group. The weights in this example sum to one, so stakeholder contributions form a weighted average before penalties. A **hard constraint** is treated as a non-negotiable eligibility condition. A **soft constraint** expresses a preference: violating it incurs a score penalty but does not automatically make a scenario ineligible.

Let $S$ be the set of scenarios, $G$ the set of stakeholder groups, and $K$ the set of ethical constraints. For scenario $s$, let $w_g$ be stakeholder $g$'s weight and $q_{sg}$ its stakeholder score. The base score is

$$B_s = \sum_{g \in G} w_g q_{sg}.$$

If scenario $s$ violates any hard constraint, its final score is zero. Otherwise, its final score is

$$V_s = \min\!\left(1,\;\max\!\left(0,\;B_s - \lambda n_s\right)\right),$$

where $n_s$ is the number of soft violations, including the minimum-carbon-reduction rule, and $\lambda=0.1$ is the penalty per violation. The outer bounds keep the reported score between zero and one. The implementation computes $q_{sg}$ from the scenario's impacts using the stakeholder-specific formulas shown in the scoring function.

The method evaluates each scenario independently and then sorts the results by $V_s$. With $|S|$ scenarios, $|G|$ stakeholder groups, and $|K|$ constraints, the evaluation takes $O(|S|(|G|+|K|))$ time; sorting adds $O(|S|\log |S|)$. This is a finite scenario comparison, not a search over every possible operational plan.

### Data used in the example

The scenario records below supply carbon reductions, cost changes, and impact scores. The scenario scores and compliance labels are calculated by the notebook rather than taken as precomputed results.

| Scenario | Carbon reduction (kg CO$_2$e/day) | Cost change (%) | Community health | Worker security | Shareholder returns | Environmental justice | Long-term sustainability |
|---|---:|---:|---:|---:|---:|---:|---:|
| Pure Carbon Optimization | 450 | -5.0 | 0.65 | -0.40 | 0.30 | 0.20 | 0.40 |
| Balanced Transition | 354 | 2.3 | 0.55 | 0.25 | 0.10 | 0.45 | 0.75 |
| Minimal Change | 129 | 0.8 | 0.20 | 0.80 | -0.10 | 0.15 | 0.30 |
| Community-First Approach | 280 | 4.5 | 0.80 | 0.10 | -0.15 | 0.85 | 0.70 |
| Worker Protection Focus | 180 | 3.2 | 0.35 | 0.90 | -0.10 | 0.40 | 0.50 |

### What to examine

- Which scenarios satisfy every hard and soft constraint, and which constraints they violate.
- How stakeholder weights and impact scores contribute to each value-aligned score.
- How the highest-scoring scenario trades carbon reduction against stakeholder impacts and cost.
- Why the recommendation is conditional on the specified values, thresholds, and scoring rule.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class EthicalPrinciple(Enum):
    """Ethical principles represented by the scenario constraints."""

    ENVIRONMENTAL_JUSTICE = "environmental_justice"
    INTERGENERATIONAL_EQUITY = "intergenerational_equity"
    WORKER_PROTECTION = "worker_protection"
    COMMUNITY_HEALTH = "community_health"
    ECONOMIC_SUSTAINABILITY = "economic_sustainability"
    TRANSPARENCY = "transparency"


@dataclass
class StakeholderGroup:
    """A stakeholder group, its relative weight, and its stated values."""

    name: str
    weight: float
    values: Dict[str, float]
    impact_sensitivity: float


@dataclass
class EthicalConstraint:
    """A threshold associated with an ethical principle."""

    principle: EthicalPrinciple
    threshold: float
    is_hard_constraint: bool
    description: str


@dataclass
class ScenarioImpact:
    """Scenario impacts; all impact fields are dimensionless scores."""

    carbon_reduction: float  # kg CO2e/day
    community_health: float
    worker_security: float
    shareholder_returns: float
    environmental_justice: float
    long_term_sustainability: float


@dataclass
class EthicalScenario:
    """An alternative carbon-reduction scenario and its calculated results."""

    id: str
    name: str
    description: str
    cost_increase: float  # Percent; negative values indicate a cost decrease.
    implementation_time: str
    impact: ScenarioImpact
    value_aligned_score: float
    compliance_status: str

In [ ]:
class ValueAlignedOptimization:
    """Evaluate scenarios using ethical constraints and weighted stakeholder scores."""

    def __init__(self) -> None:
        self.stakeholders = self._initialize_stakeholders()
        self.constraints = self._initialize_constraints()
        self.scenarios: List[EthicalScenario] = []

    def _initialize_stakeholders(self) -> List[StakeholderGroup]:
        """Define stakeholder weights and the values used to describe each group."""
        return [
            StakeholderGroup(
                name="Community",
                weight=0.35,
                values={
                    "air_quality": 0.8,
                    "noise_pollution": 0.6,
                    "green_spaces": 0.7,
                    "property_values": 0.5,
                },
                impact_sensitivity=0.8,
            ),
            StakeholderGroup(
                name="Workers",
                weight=0.30,
                values={
                    "job_security": 0.9,
                    "wage_levels": 0.7,
                    "working_conditions": 0.8,
                    "training_opportunities": 0.6,
                },
                impact_sensitivity=0.9,
            ),
            StakeholderGroup(
                name="Shareholders",
                weight=0.15,
                values={
                    "profitability": 0.8,
                    "market_share": 0.6,
                    "risk_management": 0.7,
                    "brand_reputation": 0.5,
                },
                impact_sensitivity=0.6,
            ),
            StakeholderGroup(
                name="Environment",
                weight=0.20,
                values={
                    "carbon_reduction": 0.9,
                    "biodiversity": 0.8,
                    "water_quality": 0.7,
                    "soil_health": 0.6,
                },
                impact_sensitivity=0.7,
            ),
        ]

    def _initialize_constraints(self) -> List[EthicalConstraint]:
        """Set hard eligibility rules and soft preferences for this example."""
        return [
            EthicalConstraint(
                principle=EthicalPrinciple.WORKER_PROTECTION,
                threshold=-0.2,
                is_hard_constraint=True,
                description="Worker security must be at least -0.20.",
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.ENVIRONMENTAL_JUSTICE,
                threshold=0.3,
                is_hard_constraint=True,
                description="Environmental justice impact must be at least 0.30.",
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.COMMUNITY_HEALTH,
                threshold=0.1,
                is_hard_constraint=False,
                description="Community health impact should be at least 0.10.",
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.ECONOMIC_SUSTAINABILITY,
                threshold=-0.5,
                is_hard_constraint=False,
                description="Cost increase should not exceed 50 percent.",
            ),
        ]

    def create_scenarios(self) -> List[EthicalScenario]:
        """Create the five alternatives and their supplied impact data."""
        scenarios = [
            EthicalScenario(
                id="pure_optimization",
                name="Pure Carbon Optimization",
                description="Maximum carbon reduction without ethical considerations.",
                cost_increase=-5.0,
                implementation_time="short_term",
                impact=ScenarioImpact(
                    carbon_reduction=450,
                    community_health=0.65,
                    worker_security=-0.40,
                    shareholder_returns=0.30,
                    environmental_justice=0.20,
                    long_term_sustainability=0.40,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
            ),
            EthicalScenario(
                id="balanced_transition",
                name="Balanced Transition",
                description="Carbon reduction with worker transition programs.",
                cost_increase=2.3,
                implementation_time="medium_term",
                impact=ScenarioImpact(
                    carbon_reduction=354,
                    community_health=0.55,
                    worker_security=0.25,
                    shareholder_returns=0.10,
                    environmental_justice=0.45,
                    long_term_sustainability=0.75,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
            ),
            EthicalScenario(
                id="minimal_change",
                name="Minimal Change",
                description="A conservative approach with limited disruption.",
                cost_increase=0.8,
                implementation_time="short_term",
                impact=ScenarioImpact(
                    carbon_reduction=129,
                    community_health=0.20,
                    worker_security=0.80,
                    shareholder_returns=-0.10,
                    environmental_justice=0.15,
                    long_term_sustainability=0.30,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
            ),
            EthicalScenario(
                id="community_first",
                name="Community-First Approach",
                description="Prioritize community benefits and environmental justice.",
                cost_increase=4.5,
                implementation_time="long_term",
                impact=ScenarioImpact(
                    carbon_reduction=280,
                    community_health=0.80,
                    worker_security=0.10,
                    shareholder_returns=-0.15,
                    environmental_justice=0.85,
                    long_term_sustainability=0.70,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
            ),
            EthicalScenario(
                id="worker_protection",
                name="Worker Protection Focus",
                description="Emphasize worker protection with moderate carbon goals.",
                cost_increase=3.2,
                implementation_time="short_term",
                impact=ScenarioImpact(
                    carbon_reduction=180,
                    community_health=0.35,
                    worker_security=0.90,
                    shareholder_returns=-0.10,
                    environmental_justice=0.40,
                    long_term_sustainability=0.50,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
            ),
        ]
        self.scenarios = scenarios
        return scenarios

    def _constraint_violation_message(
        self, scenario: EthicalScenario, constraint: EthicalConstraint
    ) -> Optional[str]:
        """Return an explanation when a scenario fails one specified threshold."""
        principle = constraint.principle

        if principle == EthicalPrinciple.WORKER_PROTECTION:
            value = scenario.impact.worker_security
            if value < constraint.threshold:
                return (
                    f"Worker protection: {value:.2f} is below "
                    f"{constraint.threshold:.2f}."
                )

        elif principle == EthicalPrinciple.ENVIRONMENTAL_JUSTICE:
            value = scenario.impact.environmental_justice
            if value < constraint.threshold:
                return (
                    f"Environmental justice: {value:.2f} is below "
                    f"{constraint.threshold:.2f}."
                )

        elif principle == EthicalPrinciple.COMMUNITY_HEALTH:
            value = scenario.impact.community_health
            if value < constraint.threshold:
                return (
                    f"Community health: {value:.2f} is below "
                    f"{constraint.threshold:.2f}."
                )

        elif principle == EthicalPrinciple.ECONOMIC_SUSTAINABILITY:
            maximum_cost_increase = abs(constraint.threshold) * 100
            if scenario.cost_increase > maximum_cost_increase:
                return (
                    f"Economic sustainability: cost increase of "
                    f"{scenario.cost_increase:.1f}% exceeds "
                    f"{maximum_cost_increase:.1f}%."
                )

        return None

    def check_ethical_compliance(
        self, scenario: EthicalScenario
    ) -> Tuple[str, List[str]]:
        """Check all thresholds and report each violated rule."""
        violations = []

        for constraint in self.constraints:
            message = self._constraint_violation_message(scenario, constraint)
            if message is not None:
                violations.append(message)

        minimum_carbon_reduction = 100
        if scenario.impact.carbon_reduction < minimum_carbon_reduction:
            violations.append(
                "Carbon reduction: "
                f"{scenario.impact.carbon_reduction:.0f} kg CO2e/day is below "
                f"the {minimum_carbon_reduction} kg CO2e/day minimum."
            )

        status = "VIOLATION" if violations else "FULL_COMPLIANCE"
        return status, violations

    def calculate_value_aligned_score(self, scenario: EthicalScenario) -> float:
        """Calculate the weighted score, applying hard and soft constraint rules."""
        hard_constraint_violations = []
        soft_constraint_violations = []

        for constraint in self.constraints:
            message = self._constraint_violation_message(scenario, constraint)
            if message is None:
                continue
            if constraint.is_hard_constraint:
                hard_constraint_violations.append(message)
            else:
                soft_constraint_violations.append(message)

        if hard_constraint_violations:
            return 0.0

        minimum_carbon_reduction = 100
        if scenario.impact.carbon_reduction < minimum_carbon_reduction:
            soft_constraint_violations.append(
                "The minimum carbon-reduction requirement is not met."
            )

        stakeholder_scores = []
        for stakeholder in self.stakeholders:
            if stakeholder.name == "Community":
                score = (
                    scenario.impact.community_health * 0.4
                    + scenario.impact.environmental_justice * 0.4
                    + scenario.impact.long_term_sustainability * 0.2
                )
            elif stakeholder.name == "Workers":
                score = scenario.impact.worker_security
            elif stakeholder.name == "Shareholders":
                score = (
                    scenario.impact.shareholder_returns * 0.6
                    + (1 - abs(scenario.cost_increase) / 100) * 0.4
                )
            elif stakeholder.name == "Environment":
                score = (
                    scenario.impact.carbon_reduction / 450 * 0.5
                    + scenario.impact.environmental_justice * 0.3
                    + scenario.impact.long_term_sustainability * 0.2
                )
            else:
                raise ValueError(f"No scoring rule is defined for {stakeholder.name}.")

            stakeholder_scores.append(score * stakeholder.weight)

        base_score = sum(stakeholder_scores)
        penalty = len(soft_constraint_violations) * 0.1
        return min(1.0, max(0.0, base_score - penalty))

    def evaluate_scenarios(self) -> List[EthicalScenario]:
        """Calculate and store each scenario's compliance status and score."""
        for scenario in self.scenarios:
            scenario.compliance_status, _ = self.check_ethical_compliance(scenario)
            scenario.value_aligned_score = self.calculate_value_aligned_score(scenario)
        return self.scenarios


ethical_optimizer = ValueAlignedOptimization()
scenarios = ethical_optimizer.create_scenarios()
scenarios = ethical_optimizer.evaluate_scenarios()

assert abs(sum(group.weight for group in ethical_optimizer.stakeholders) - 1.0) < 1e-12
assert len(scenarios) == 5
assert all(0.0 <= scenario.value_aligned_score <= 1.0 for scenario in scenarios)

print(f"Created and evaluated {len(scenarios)} scenarios.")
print(f"Stakeholder groups: {[group.name for group in ethical_optimizer.stakeholders]}")
print(f"Ethical constraints: {len(ethical_optimizer.constraints)}")

## Scenario evaluation and recommendation

The compliance check compares each scenario with the thresholds defined above. A violation of a hard constraint sets the scenario's score to zero. A soft-constraint violation instead reduces the weighted stakeholder score by 0.1 for each violation. The carbon-reduction minimum is also included in the penalty count. The ranking is therefore determined by the stated scoring rule, not by carbon reduction alone.

The code displays the score, compliance status, cost change, carbon reduction, and any violated rules. The recommendation is the scenario with the highest calculated score. If two or more scores tie, the stable sort preserves their order in the input data; a tie should be understood as multiple equally ranked options under this model.

In [ ]:
def analyze_ethical_scenarios(
    scenarios: List[EthicalScenario],
    optimizer: ValueAlignedOptimization,
) -> Dict[str, object]:
    """Rank scenarios, explain constraint results, and return the recommendation."""
    if not scenarios:
        raise ValueError("At least one scenario is required for analysis.")

    sorted_scenarios = sorted(
        scenarios,
        key=lambda scenario: scenario.value_aligned_score,
        reverse=True,
    )

    print("\n" + "=" * 88)
    print("ETHICAL SCENARIO EVALUATION RESULTS")
    print("=" * 88)
    print("\nSCENARIO RANKING")
    print(
        f"{'Rank':<6}{'Scenario':<32}{'Score':>9}  "
        f"{'Compliance':<18}{'Carbon reduction':>20}"
    )
    print("-" * 88)

    for rank, scenario in enumerate(sorted_scenarios, start=1):
        print(
            f"{rank:<6}{scenario.name:<32}{scenario.value_aligned_score:>9.3f}  "
            f"{scenario.compliance_status:<18}"
            f"{scenario.impact.carbon_reduction:>10.0f} kg CO2e/day"
        )

    print("\nSCENARIO DETAILS")
    for scenario in sorted_scenarios:
        print(f"\n{scenario.name}")
        print(f"  Description: {scenario.description}")
        print(f"  Value-aligned score: {scenario.value_aligned_score:.3f}")
        print(f"  Compliance: {scenario.compliance_status}")
        print(f"  Cost change: {scenario.cost_increase:+.1f}%")
        print(f"  Implementation period: {scenario.implementation_time.replace('_', ' ')}")
        print(f"  Carbon reduction: {scenario.impact.carbon_reduction:.0f} kg CO2e/day")

        _, violations = optimizer.check_ethical_compliance(scenario)
        if violations:
            print("  Constraint violations:")
            for violation in violations:
                print(f"    - {violation}")
        else:
            print("  Constraint violations: None")

    best_scenario = sorted_scenarios[0]
    tied_scenarios = [
        scenario
        for scenario in sorted_scenarios
        if np.isclose(
            scenario.value_aligned_score,
            best_scenario.value_aligned_score,
            rtol=1e-9,
            atol=1e-12,
        )
    ]

    print("\nRECOMMENDATION UNDER THE SPECIFIED SCORING RULE")
    print(f"Highest-ranked scenario: {best_scenario.name}")
    print(f"Value-aligned score: {best_scenario.value_aligned_score:.3f}")
    print(f"Compliance: {best_scenario.compliance_status}")
    print(f"Carbon reduction: {best_scenario.impact.carbon_reduction:.0f} kg CO2e/day")
        
    if len(tied_scenarios) > 1:
        tied_names = ", ".join(scenario.name for scenario in tied_scenarios)
        print(f"Scenarios tied for the highest score: {tied_names}")
    print("This ranking is conditional on the weights, thresholds, and score definitions in this model.")

    return {
        "best_scenario": best_scenario,
        "all_scenarios": sorted_scenarios,
        "ethical_compliance": {
            scenario.id: scenario.compliance_status for scenario in scenarios
        },
    }


analysis_results = analyze_ethical_scenarios(scenarios, ethical_optimizer)

In [ ]:
def visualize_ethical_analysis(
    scenarios: List[EthicalScenario],
    analysis_results: Dict[str, object],
) -> None:
    """Plot scores, impact trade-offs, stakeholder profiles, and cost outcomes."""
    if not scenarios:
        raise ValueError("At least one scenario is required for visualization.")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Value-Aligned and Ethical Framework Analysis", fontsize=16, fontweight="bold")

    # Bar colors encode whether a scenario satisfies every stated constraint.
    ax1 = axes[0, 0]
    names = [scenario.name for scenario in scenarios]
    scores = [scenario.value_aligned_score for scenario in scenarios]
    colors = [
        "seagreen" if scenario.compliance_status == "FULL_COMPLIANCE" else "indianred"
        for scenario in scenarios
    ]
    bars = ax1.bar(names, scores, color=colors, alpha=0.8, edgecolor="black")
    ax1.set_xlabel("Scenario")
    ax1.set_ylabel("Value-aligned score")
    ax1.set_title("Calculated score by scenario")
    ax1.set_ylim(0, 1.08)
    ax1.grid(True, alpha=0.3, axis="y")
    ax1.tick_params(axis="x", rotation=35)
    for label in ax1.get_xticklabels():
        label.set_ha("right")
    for bar, score in zip(bars, scores):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{score:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax1.text(
        0.02,
        0.98,
        "Green: full compliance; red: at least one violation",
        transform=ax1.transAxes,
        va="top",
        fontsize=8,
    )

    ax2 = axes[0, 1]
    carbon_reductions = [scenario.impact.carbon_reduction for scenario in scenarios]
    worker_impacts = [scenario.impact.worker_security for scenario in scenarios]
    community_impacts = [scenario.impact.community_health for scenario in scenarios]
    scatter = ax2.scatter(
        carbon_reductions,
        worker_impacts,
        c=community_impacts,
        s=110,
        alpha=0.8,
        cmap="RdYlGn",
        edgecolors="black",
    )
    for scenario in scenarios:
        ax2.annotate(
            scenario.name,
            (scenario.impact.carbon_reduction, scenario.impact.worker_security),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
    ax2.set_xlabel("Carbon reduction (kg CO2e/day)")
    ax2.set_ylabel("Worker security impact score")
    ax2.set_title("Carbon reduction and stakeholder impacts")
    ax2.grid(True, alpha=0.3)
    colorbar = fig.colorbar(scatter, ax=ax2)
    colorbar.set_label("Community health impact score")

    ax3 = fig.add_subplot(2, 2, 3, projection="polar")
    top_scenarios = analysis_results["all_scenarios"][:3]
    categories = [
        "Community health",
        "Worker security",
        "Shareholder returns",
        "Environmental justice",
        "Long-term sustainability",
    ]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    closed_angles = angles + angles[:1]
    radar_colors = ["tab:blue", "tab:green", "tab:orange"]
    for index, scenario in enumerate(top_scenarios):
        values = [
            scenario.impact.community_health,
            scenario.impact.worker_security,
            scenario.impact.shareholder_returns,
            scenario.impact.environmental_justice,
            scenario.impact.long_term_sustainability,
        ]
        closed_values = values + values[:1]
        color = radar_colors[index % len(radar_colors)]
        ax3.plot(closed_angles, closed_values, "o-", linewidth=2, label=scenario.name, color=color)
        ax3.fill(closed_angles, closed_values, alpha=0.15, color=color)
    ax3.set_xticks(angles)
    ax3.set_xticklabels(categories, fontsize=8)
    ax3.set_ylim(-1, 1)
    ax3.set_title("Stakeholder impact profiles of the top-ranked scenarios", pad=22)
    ax3.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)
    ax3.grid(True)

    ax4 = axes[1, 1]
    marker_by_status = {"FULL_COMPLIANCE": "o", "VIOLATION": "X"}
    color_by_status = {"FULL_COMPLIANCE": "seagreen", "VIOLATION": "indianred"}
    for status in ("FULL_COMPLIANCE", "VIOLATION"):
        selected = [
            scenario for scenario in scenarios if scenario.compliance_status == status
        ]
        if selected:
            ax4.scatter(
                [scenario.cost_increase for scenario in selected],
                [scenario.impact.carbon_reduction for scenario in selected],
                s=100,
                color=color_by_status[status],
                marker=marker_by_status[status],
                edgecolors="black",
                alpha=0.8,
                label=status.replace("_", " ").title(),
            )
    for scenario in scenarios:
        ax4.annotate(
            scenario.name,
            (scenario.cost_increase, scenario.impact.carbon_reduction),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
    ax4.set_xlabel("Cost change (%)")
    ax4.set_ylabel("Carbon reduction (kg CO2e/day)")
    ax4.set_title("Cost and carbon reduction by compliance status")
    ax4.grid(True, alpha=0.3)
    ax4.legend()

    fig.tight_layout()
    plt.show()


visualize_ethical_analysis(scenarios, analysis_results)

## Interpreting the plots

The score chart summarizes the weighted rule-based evaluation; its colors indicate whether all stated constraints are satisfied. In the scatter plot, each point is a scenario, its horizontal position is carbon reduction, its vertical position is worker-security impact, and its color represents community-health impact. The radar chart compares the five supplied impact dimensions for the highest-ranked scenarios. The final scatter plot shows cost change against carbon reduction and uses markers to identify compliance status.

These visualizations show the supplied scenario scores and modeled relationships. They do not establish that a proposed intervention will cause the represented impacts in a real logistics network.

In [ ]:
def ethical_framework_analysis(
    optimizer: ValueAlignedOptimization,
    scenarios: List[EthicalScenario],
    analysis_results: Dict[str, object],
) -> None:
    """Summarize the model structure, compliance results, and recommendation."""
    print("=" * 80)
    print("VALUE-ALIGNED AND ETHICAL FRAMEWORK ANALYSIS")
    print("=" * 80)

    print("\nFRAMEWORK COMPONENTS")
    print("- Weighted stakeholder scoring")
    print("- Hard eligibility constraints and soft-constraint penalties")
    print("- Explicit reporting of scenario impacts and constraint violations")
    print("- A ranked recommendation based on the defined value-aligned score")

    print("\nSTAKEHOLDER GROUPS")
    for stakeholder in optimizer.stakeholders:
        print(f"{stakeholder.name} (weight: {stakeholder.weight:.2f})")
        print(f"  Impact sensitivity: {stakeholder.impact_sensitivity:.2f}")
        print(f"  Stated values: {', '.join(stakeholder.values)}")

    print("\nETHICAL CONSTRAINTS")
    for constraint in optimizer.constraints:
        constraint_type = "Hard" if constraint.is_hard_constraint else "Soft"
        print(f"{constraint.principle.value.replace('_', ' ').title()} ({constraint_type})")
        print(f"  Threshold: {constraint.threshold:.2f}")
        print(f"  Rule: {constraint.description}")

    compliant = [
        scenario
        for scenario in scenarios
        if scenario.compliance_status == "FULL_COMPLIANCE"
    ]
    violating = [
        scenario for scenario in scenarios if scenario.compliance_status == "VIOLATION"
    ]
    print("\nCOMPLIANCE SUMMARY")
    print(f"Scenarios evaluated: {len(scenarios)}")
    print(f"Fully compliant: {len(compliant)}")
    print(f"With one or more violations: {len(violating)}")

    for scenario in violating:
        _, violations = optimizer.check_ethical_compliance(scenario)
        print(f"  {scenario.name}:")
        for violation in violations:
            print(f"    - {violation}")

    best_scenario = analysis_results["best_scenario"]
    print("\nHIGHEST-RANKED SCENARIO")
    print(f"Scenario: {best_scenario.name}")
    print(f"Value-aligned score: {best_scenario.value_aligned_score:.3f}")
    print(f"Compliance: {best_scenario.compliance_status}")
    print(f"Carbon reduction: {best_scenario.impact.carbon_reduction:.0f} kg CO2e/day")
    print(
        "The result is conditional on the input data, stakeholder weights, "
        "thresholds, and scoring formulas."
    )

    print("\nPRACTICAL CONSIDERATIONS")
    print("- Elicit and document stakeholder weights transparently.")
    print("- Justify impact scales and thresholds with appropriate evidence.")
    print("- Monitor realized outcomes and revise assumptions through governance.")
    print("- Communicate trade-offs and uncertainty alongside any recommendation.")
    print("- Treat the score as a decision aid rather than a substitute for deliberation.")


ethical_framework_analysis(ethical_optimizer, scenarios, analysis_results)

## Following one scenario through the decision process

The next cell examines the highest-scoring scenario using the same data and rules as the ranking. It reports stakeholder impacts, constraint checks, score, and trade-offs directly from the evaluated scenario. This makes the recommendation auditable: a reader can trace it from supplied impact data through the rules to the final ranking.

In [ ]:
def demonstrate_ethical_decision_making(
    optimizer: ValueAlignedOptimization,
    scenarios: List[EthicalScenario],
    analysis_results: Dict[str, object],
) -> None:
    """Trace the highest-scoring scenario through the model's decision steps."""
    best_scenario = analysis_results["best_scenario"]
    pure_optimization = next(
        (scenario for scenario in scenarios if scenario.id == "pure_optimization"),
        None,
    )

    print("=" * 80)
    print("ETHICAL DECISION-MAKING DEMONSTRATION")
    print("=" * 80)
    print(f"\nScenario under examination: {best_scenario.name}")

    print("\n1. STAKEHOLDER IMPACT ASSESSMENT")
    impact_by_group = {
        "Community": best_scenario.impact.community_health,
        "Workers": best_scenario.impact.worker_security,
        "Shareholders": best_scenario.impact.shareholder_returns,
        "Environment": best_scenario.impact.carbon_reduction / 450,
    }
    for stakeholder in optimizer.stakeholders:
        impact = impact_by_group[stakeholder.name]
        interpretation = "positive" if impact > 0 else "negative" if impact < 0 else "neutral"
        print(f"  {stakeholder.name}: {impact:+.2f} ({interpretation} on the modeled scale)")
    print("  The environmental value shown here is normalized by 450 kg CO2e/day.")

    print("\n2. ETHICAL CONSTRAINT CHECK")
    status, violations = optimizer.check_ethical_compliance(best_scenario)
    print(f"  Overall status: {status}")
    if violations:
        for violation in violations:
            print(f"  Violation: {violation}")
    else:
        print("  All modeled thresholds are satisfied.")

    print("\n3. VALUE-ALIGNED SCORE")
    print(f"  Calculated score: {best_scenario.value_aligned_score:.3f}")
    print("  The score combines stakeholder-weighted impacts and any soft-constraint penalties.")

    print("\n4. TRADE-OFFS")
    print(f"  Carbon reduction: {best_scenario.impact.carbon_reduction:.0f} kg CO2e/day")
    print(f"  Cost change: {best_scenario.cost_increase:+.1f}%")
    print(f"  Implementation period: {best_scenario.implementation_time.replace('_', ' ')}")
    if pure_optimization is not None:
        carbon_difference = (
            pure_optimization.impact.carbon_reduction
            - best_scenario.impact.carbon_reduction
        )
        worker_difference = (
            best_scenario.impact.worker_security
            - pure_optimization.impact.worker_security
        )
        print("  Comparison with Pure Carbon Optimization:")
        print(f"    Carbon reduction difference: {carbon_difference:+.0f} kg CO2e/day")
        print(f"    Worker security impact difference: {worker_difference:+.2f}")

    print("\n5. DECISION INTERPRETATION")
    print(
        f"  {best_scenario.name} ranks highest under the specified score and constraints."
    )
    print(
        "  This does not establish that it is universally preferable; different "
        "stakeholder weights or justified thresholds could change the ranking."
    )

    print("\n6. IMPLEMENTATION QUESTIONS")
    print("  - Which stakeholders should validate the impact estimates and weights?")
    print("  - How will worker, community, and environmental outcomes be monitored?")
    print("  - What governance process will review outcomes against the stated thresholds?")
    print("  - How will uncertainty and distributional effects be reported?")


demonstrate_ethical_decision_making(
    ethical_optimizer,
    scenarios,
    analysis_results,
)

## Role, strengths, and limitations of the framework

### Role

A carbon-reduction decision can affect communities, workers, shareholders, and the environment in different ways. This framework makes those modeled concerns visible in the decision process by combining stakeholder-weighted scores with explicit hard and soft constraints. It complements technical evaluation by making the chosen value judgments inspectable; it does not remove the need for stakeholder engagement or evidence-based impact assessment.

### Strengths

- Makes stakeholder weights, impact scores, constraints, and penalties explicit.
- Can rule out scenarios that violate specified hard requirements.
- Shows how a weighted score and soft-constraint penalties shape a ranking.
- Provides scenario-level explanations that can be examined and challenged.
- Helps structure discussion of environmental justice, worker protection, and long-term effects.

### Limitations

- Weights and impact scores encode judgments and may be contested.
- A single weighted score can conceal disagreements between stakeholder groups.
- The supplied scenario impacts are not independently measured or causally validated here.
- The score does not represent uncertainty, distribution within stakeholder groups, or changing impacts over time.
- A high score is not proof of legal compliance, social acceptance, or ethical legitimacy.
- Comparing scenario values depends on the assumed scales and normalization choices.

### Appropriate uses

This type of framework can help organize early-stage discussion for corporate sustainability planning, infrastructure assessment, supply-chain decisions, and public policy. In consequential applications, the scenario data, thresholds, value scales, and governance process should be validated with affected stakeholders and relevant evidence.

### Summary

The notebook evaluates a finite set of carbon-reduction scenarios using a specified sequence: check ethical constraints, calculate stakeholder-weighted scores with penalties, rank the scenarios, and explain the highest-ranked result. The recommendation is meaningful only relative to those inputs and rules. Its main educational lesson is that ethical constraints and stakeholder values must be made explicit if they are to influence a quantitative decision.